In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

import skimage
from skimage.measure import label
from skimage.filters import threshold_otsu
from skimage.morphology import remove_small_objects
from skimage.morphology import remove_small_holes
from skimage.morphology import disk

from scipy.ndimage import binary_opening
from scipy.ndimage import binary_dilation
from scipy.ndimage import distance_transform_edt
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter


from tqdm import tqdm

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# Import images and csv files

runs_dir = Path(r'/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Puro10min')

cytoplasm_dir = runs_dir / 'ROIs_corrected'
cytoplasm_path = os.path.join(cytoplasm_dir, '**', '*.tif')  # Use '**' to match subdirectories
cytoplasm_files = np.sort(glob.glob(cytoplasm_path, recursive=True))

# Extract the integers before the first underscore in the filenames
loaded_image_ids = {
    os.path.basename(f).split('_')[0]  # Extract the part before the first underscore
    for f in cytoplasm_files
}

nucleus_dir = runs_dir / 'Nucleus'
nucleus_path = os.path.join(nucleus_dir,'*.tif')
nucleus_files = np.sort(glob.glob(nucleus_path))

# Filter the new images based on matching IDs
nucleus_files = np.array([
    f for f in nucleus_files
    if os.path.basename(f).split('_')[0] in loaded_image_ids
])

actin_dir = runs_dir / 'Actin'
actin_path = os.path.join(actin_dir,'*.tif')
actin_files = np.sort(glob.glob(actin_path))

# Filter the new images based on matching IDs
actin_files = np.array([
    f for f in actin_files
    if os.path.basename(f).split('_')[0] in loaded_image_ids
])

tubulin_dir = runs_dir / 'Tubulin'
tubulin_path = os.path.join(tubulin_dir,'*.tif')
tubulin_files = np.sort(glob.glob(tubulin_path))

# Filter the new images based on matching IDs
tubulin_files = np.array([
    f for f in tubulin_files
    if os.path.basename(f).split('_')[0] in loaded_image_ids
])

tracks_dir = runs_dir / 'Factor6Link5'
tracks_path = os.path.join(tracks_dir,'*.csv')
tracks_files = np.sort(glob.glob(tracks_path))

In [ ]:
print(len(cytoplasm_files))
print(len(nucleus_files))
print(len(actin_files))
print(len(tubulin_files))
print(len(tracks_files))

In [ ]:
# Read images and csv files into lists

actin_images = []

for file in actin_files:
    image = imread(file)
    actin_images.append(image)

tubulin_images = []

for file in tubulin_files:
    image = imread(file)
    tubulin_images.append(image)

cytoplasm_images = []

for file in cytoplasm_files:
    image = imread(file)
    cytoplasm_images.append(image)

nucleus_images = []

for file in nucleus_files:
    image = imread(file)
    nucleus_images.append(image)

tracks = []

for file in tracks_files:
    tracks_all = pd.read_csv(file)
    tracks.append(tracks_all)

for i in range(len(tracks)):
    tracks[i] = tracks[i].loc[:, ~tracks[i].columns.str.contains('^Unnamed')]

tracks[0].head()

#### Creating composite images of cytoskeleton and thresholding them to create masks

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (12,4))
ax[0].imshow(actin_images[-1])
ax[1].imshow(tubulin_images[-1])
ax[2].imshow(cytoplasm_images[-1])

In [ ]:
# Merge actin and tubulin images

merged_images = []

for img1, img2 in zip(actin_images, tubulin_images):
    # Add images on top of each other
    merged_img = img1 + img2

    merged_images.append(merged_img)

print(len(merged_images))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (12,6))
ax[0].imshow(merged_images[0])
ax[1].imshow(cytoplasm_images[0])

In [ ]:
# Creates a stack and calculates the global otsu threshold which will be used to threshold all images

merged_stack = np.hstack(merged_images)
merged_otsu = threshold_otsu(merged_stack)*0.3 # Factor to manually adjust otsu
print(merged_otsu)

In [ ]:
# Segments merged_images based on merged_otsu

cytoskel_masks = []
for img in merged_images:
    mask = (img >= merged_otsu) # Binary mask
    #print(merged_otsu)
    cytoskel_masks.append(mask)

for i in range(len(cytoskel_masks)):
    mask_label = label(cytoskel_masks[i].astype(np.int64)) # Mask needs to be converted to integers for following operations
    cytoskel_masks[i] = remove_small_objects(mask_label, min_size=1000)  # Remove small objects
    cytoskel_masks[i] = (remove_small_holes(cytoskel_masks[i], area_threshold=2000) > 0)
    cytoskel_masks[i] = gaussian_filter(cytoskel_masks[i].astype(float), sigma=1.5) > 0.5 # Smoothen edges

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (12,6))
ax[0].imshow(merged_images[-1])
ax[1].imshow(cytoskel_masks[-1])

#### Using cytoskel_masks to identify "spikes"

In [459]:
# Creating spikes

masks_spikes = []

for image in cytoskel_masks:    
    opened_image = binary_opening(image, iterations=10) # Opens the image to smoothen out the spikes
    spikes = image ^ opened_image # Creates the spikes by substracting the opened image from the entire mask (boolean operator)


    # Refining spike mask by removing small objects and "large" objects, i.e. misidentified neurites
    refined_spikes = label(spikes.astype(np.int64))
    refined_spikes = remove_small_objects(refined_spikes, min_size=30)  # Remove small objects
    obj_trash = remove_small_objects(refined_spikes, min_size=500)  # Define large objects to remove in the next step
    refined_spikes = (refined_spikes ^ obj_trash) > 0 # Large objects are substracted from entire mask to leave only the spikes
    masks_spikes.append(refined_spikes)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (12,6))
ax[0].imshow(masks_spikes[2])
ax[1].imshow(merged_images[2])

#### Merging cytoplasm mask with nucleus

In [461]:
# Adding nuclei back to cytplasm mask

whole_cells = []

for cyto_mask, nucleus_mask in zip(cytoplasm_images, nucleus_images):

    # Define maximum expansion in pixels
    max_expansion = 20  # Maximum expansion distance in pixels

    # Ensure masks have the same shape
    assert cyto_mask.shape == nucleus_mask.shape, "Masks must have the same dimensions"

    # Get unique nucleus labels
    nucleus_labels = np.unique(nucleus_mask)
    nucleus_labels = nucleus_labels[nucleus_labels > 0]  # Remove background (0)

    # Create a new expanded nucleus mask
    expanded_nucleus_mask = np.copy(nucleus_mask)

    for label_value in nucleus_labels:
        # Extract nucleus region
        nucleus_region = (nucleus_mask == label_value)
        
        # Extract corresponding cytoplasm region
        cyto_region = (cyto_mask == label_value)

        # Initialize dilation step
        dilation_step = 1
        expanded_region = nucleus_region.copy()  # Start with the original nucleus region
        
        while dilation_step <= max_expansion:
            # Apply dilation with a small disk (disk(1) to expand step by step)
            expanded_region = binary_dilation(expanded_region, disk(1))  # Expand by 1 pixel per step
            
            # Check if **all** of the expanded region touches the cytoplasm
            if np.all(expanded_region & cyto_region):  # Stop if ALL of the expanded region touches cytoplasm
                print(f"Expansion stopped at {dilation_step} pixels due to touching cytoplasm.")
                break  # Stop expanding when all of it touches the cytoplasm
            
            dilation_step += 1  # Increment the dilation step
        
        # Assign the expanded region to the output mask
        expanded_nucleus_mask[expanded_region] = label_value  # Update the expanded region in the mask

    # Merge the expanded nuclei with the cytoplasm mask
    whole_cell = np.where(expanded_nucleus_mask > 0, expanded_nucleus_mask, cyto_mask)
    whole_cells.append(whole_cell)

In [ ]:
plt.imshow(whole_cells[0])

#### Masking out cell body and neurites from cytoplasm_images (per ROI)

In [ ]:
# Seperating cell masks into soma_outgrowth and neurite

neurites_mask_binary = []
soma_outgrowth_mask_binary = []

for cell_mask in whole_cells:

    # Get unique ROI labels from the mask
    roi_labels = np.unique(cell_mask)
    roi_labels = roi_labels[roi_labels > 0]  # Exclude the background (0)

    threshold_uniform = 8  # Distance threshold
    smoothed_distance_all_rois = []  # To store results for each ROI

    # Process each ROI individually
    for label_value in roi_labels:
        # Create a mask for the current ROI
        roi_mask = (cell_mask == label_value).astype(np.uint8)
        
        # Distance transform for the current ROI
        distance_transform_body = distance_transform_edt(roi_mask)
        smoothed_distance = uniform_filter(distance_transform_body, size=64) * roi_mask
        
        # Threshold to identify soma outgrowth
        soma_outgrowth = smoothed_distance > threshold_uniform
        soma_outgrowth = (binary_dilation(soma_outgrowth, iterations=15) * roi_mask) > 0  # Dilate to smooth out soma boundary
            
        # Identify neurites by thresholding the distance transform
        neurites = (np.logical_and((smoothed_distance > 0), (smoothed_distance < threshold_uniform)).astype(int) - soma_outgrowth.astype(int)) > 0
        
        # Label the neurites and remove small objects
        neurites = remove_small_objects(label(neurites.astype(int)), min_size=150) > 0
        
        # Store the result for this ROI
        smoothed_distance_all_rois.append((label_value, smoothed_distance, soma_outgrowth, neurites))

    # Combine the results back into one mask
    neurites_mask = np.zeros_like(cell_mask)  # Neurite outgrowth mask for all ROIs
    soma_outgrowth_mask = np.zeros_like(cell_mask)  # Soma outgrowth mask for all ROIs

    for label_value, smoothed_distance, soma_outgrowth, neurites in smoothed_distance_all_rois:
        neurites_mask[neurites] = label_value  # Assign each neurite to its corresponding label in the merged mask
        neurites_mask = neurites_mask > 0 # Binarize again

    neurites_mask_binary.append(neurites_mask)

    for label_value, smoothed_distance, soma_outgrowth, neurites in smoothed_distance_all_rois:
        soma_outgrowth_mask[soma_outgrowth] = label_value  # Assign each neurite to its corresponding label in the merged mask
        soma_outgrowth_mask = soma_outgrowth_mask > 0 # Binarize again

    soma_outgrowth_mask_binary.append(soma_outgrowth_mask)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize = (15,8))
ax[0].imshow(merged_images[-1])
ax[1].imshow(soma_outgrowth_mask_binary[-1])
ax[2].imshow(neurites_mask_binary[-1])

In [465]:
# Complete labeled image

mask_compartments_prelim = []

for soma_outgrowth_final, neurite_final in zip(soma_outgrowth_mask_binary, neurites_mask_binary):
    # Create the initial mask with soma_outgrowth_mask_binary and neurites_mask_binary
    mask_compartment = soma_outgrowth_final + neurite_final * 2
    mask_compartments_prelim.append(mask_compartment)

mask_compartments = []

for prelim, spikes_final in zip(mask_compartments_prelim, masks_spikes):
    # Replace values where masks_spikes is present
    mask_compartment = np.where(spikes_final > 0, 3, prelim)
    mask_compartments.append(mask_compartment)

In [ ]:
fig, ax = plt.subplots(1,2, figsize = (10, 8))
ax[0].imshow(mask_compartments[2])
ax[1].imshow(merged_images[2])

In [ ]:
# Convert images from dtype int64 to uint16 for saving

mask_compartment_export = []

for img in mask_compartments:
    img = img.astype(np.uint16)
    mask_compartment_export.append(img)

mask_compartment_export[0].dtype

In [ ]:
# Function to assign compartments to spots

def assign_cell_compartments(tracks, mask_compartments):
    tracks_comp = []  # Store processed DataFrames

    for df, mask in tqdm(zip(tracks, mask_compartments), total=len(tracks), desc="Processing tracks"):
        df = df.sort_values(by=["frame"]).copy()  # Ensure sorting
        df["cell_comp"] = np.nan  # Initialize column

        # Set dtype of 'cell_comp' to object to avoid warnings
        df["cell_comp"] = df["cell_comp"].astype("object")

        # Iterate over unique tracks
        for unique_id, track_df in df.groupby("unique_id"):
            track_indices = track_df.index  # Keep original indices
            labels = []  # Store labels for the track

            # Assign initial labels
            for i, row in track_df.iterrows():
                x, y = int(row["x"]), int(row["y"])

                # Ensure x, y are within bounds of the mask image
                if 0 <= x < mask.shape[1] and 0 <= y < mask.shape[0]:
                    label = mask[y, x]  # Retrieve label from mask
                else:
                    label = 0  # Assign label 0 if out of bounds

                if label == 1:  # Decision based on distance to nucleus
                    category = "soma" if row["distance_nucleus"] < 5 else "outgrowth"
                elif label == 2:
                    category = "neurite"
                elif label == 3:
                    category = "spike"
                else:
                    category = None  # Placeholder for '0' values

                labels.append(category)

            # Iterative propagation of missing labels (0 values)
            labels = pd.Series(labels, index=track_indices)
            if labels.isnull().all():
                labels[:] = "nan"  # If all values are 0, set to "nan"
            else:
                # Iteratively propagate until no 'None' (previously 0) values remain
                while labels.isnull().any():
                    labels.bfill(inplace=True)  # Backward fill missing values
                    labels.ffill(inplace=True)  # Forward fill missing values                    

            # Assign final labels
            df.loc[track_indices, "cell_comp"] = labels

        # Reorder the DataFrame back to its original order using the index
        df = df.sort_index()  # This will restore the original order of rows

        tracks_comp.append(df)

    return tracks_comp  # Return the list of updated DataFrames


In [ ]:
# Usage:
tracks_comp = assign_cell_compartments(tracks, mask_compartments)
tracks_comp[0].head(10)

In [ ]:
# Saving updated csv files

# Base directory and subfolder
base_dir = os.path.dirname(tracks_dir)  # Get the directory of the current file
subfolder = "Cell_comp"

subfolder_path = os.path.join(base_dir, subfolder)
os.makedirs(subfolder_path, exist_ok=True)  # Create folder if it doesn't exist

for table, file_path in zip(tracks_comp, tracks_files):

    # Define the output file path with the correct filename
    output_file = os.path.join(subfolder_path, os.path.basename(file_path))

    # Save the modified CSV
    table.to_csv(output_file, index=False)

print(f"Tables saved in: {subfolder_path}")


In [ ]:
# Saving cell compartment images

output_dir_compartment = '/Volumes/gchao/bamfaile/Analysis/ACTB-KI/00_Batch20241011_iNs/D10/Puro10min'

# Define subfolder name
subfolder = "Cell_compartments"
subfolder_path = os.path.join(output_dir_compartment, subfolder)
os.makedirs(subfolder_path, exist_ok=True)  # Ensure the subfolder exists

for img, tiff_file in zip(mask_compartment_export, cytoplasm_files):
    # Extract the original file name without the extension
    file_name = os.path.splitext(os.path.basename(tiff_file))[0]

    # Define the output file path inside the subfolder
    output_file = os.path.join(subfolder_path, file_name + '_compartments.tif')
    print(output_file)

    # Save the manipulated image as a TIFF
    tf.imwrite(output_file, img)

print(f"Images saved in: {subfolder_path}")

#### In case distance to nucleus still needs to be calculated, this can be moved up to after reading in the images

In [ ]:
# Compute distance transform

pixel_size = 0.134  # Pixel size in micrometers
default_distance = 150  # Default distance in micrometers for ROIs without a nucleus

for csv, nucleus in zip(tracks, nucleus_files):
    csv['distance_nucleus'] = np.nan  # Create an empty column

    for roi in csv['roi_id'].unique():
        tracks_roi = csv[csv['roi_id'] == roi]
        
        # Check if the nucleus exists for the current ROI
        if roi not in np.unique(nucleus):
            # Assign the default distance for all coordinates in this ROI
            csv.loc[csv['roi_id'] == roi, 'distance_nucleus'] = default_distance
            continue
        
        # Compute the distance transform for the nucleus mask
        mask_distance_transform = scipy.ndimage.distance_transform_edt(~(nucleus == roi))
        
        # Round and clip coordinates
        coords_round = np.round(np.array(tracks_roi[['x', 'y']])).astype(int)
        coords_round[:, 0] = np.clip(coords_round[:, 0], 0, mask_distance_transform.shape[0] - 1)
        coords_round[:, 1] = np.clip(coords_round[:, 1], 0, mask_distance_transform.shape[1] - 1)
        
        # Calculate distances in micrometers
        distances_in_um = mask_distance_transform[coords_round[:, 1], coords_round[:, 0]] * pixel_size
        csv.loc[csv['roi_id'] == roi, 'distance_nucleus'] = distances_in_um
